In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Import Libraries

In [ ]:
pip install -q transformers datasets accelerate

In [ ]:
import pandas as pd
import numpy as np

from transformers import AutoTokenizer, AutoModelForMultipleChoice,TrainingArguments, Trainer, EarlyStoppingCallback
from datasets import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

import torch

In [ ]:
import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")

wandb.login(key=wandb_api_key)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# EDA

In [ ]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train_df.shape)
print(test_df.shape)

In [ ]:
train_df.head()

### Missing Value Check

In [ ]:
train_df.isnull().sum()

### Viz1: Plot count of each correct answer choice


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.countplot(x="answer", data=train_df)
plt.title("Correct Answer Counts")
plt.xlabel("Option Choice")
plt.ylabel("Total Questions")
plt.show()


### Viz2: Word count distribution

In [ ]:
train_viz = train_df.copy()

train_viz["prompt_words"] = train_viz["prompt"].str.split().str.len()

train_viz["prompt_words"].hist(bins=20, color="skyblue", edgecolor="black")
plt.title("Question Prompt Lengths")
plt.xlabel("Number of Words")
plt.ylabel("Number of Questions")
plt.show()

print("Average prompt length:", train_viz["prompt_words"].mean())
print("Max prompt length    :", train_viz["prompt_words"].max())

### Viz3: Rank of Correct Answer by length

In [ ]:
def get_answer_length_rank(row):
    lengths = {col: len(str(row[col]).split()) for col in ['A','B','C','D','E']}
    sorted_cols = sorted(lengths, key=lengths.get, reverse=True) 
    return sorted_cols.index(row["answer"]) + 1 

train_df["answer_length_rank"] = train_df.apply(get_answer_length_rank, axis=1)

sns.countplot(x="answer_length_rank", data=train_df)
plt.title("Rank of Correct Answer by Length (1 = Longest Option)")
plt.xlabel("Length Rank of Correct Answer")
plt.ylabel("Count")
plt.show()

# Train-Test Split

In [ ]:
train_df, val_df = train_test_split(train_df, test_size=0.2, stratify=train_df["answer"], random_state=4524)

# Evaluation Metric

## MAP@3
Score = 1.0 if correct answer is rank 1, 0.5 if rank 2, 1/3 if rank 3, else 0.

In [ ]:
def mean_average_precision_at_3(true_labels, top3_preds):
    scores = []
    for true_label, preds in zip(true_labels, top3_preds):
        if true_label == preds[0]:
            scores.append(1.0)
        elif true_label == preds[1]:
            scores.append(0.5)
        elif true_label == preds[2]:
            scores.append(1 / 3)
        else:
            scores.append(0.0)
    return np.mean(scores)

# Model 2: DeBERTA Fine Tune'

In [ ]:
run = wandb.init(
    project="24f3004524-t22026",
    name="deberta-run1"
)

### Load tokenizer

In [ ]:
model_name = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

### Preprocess
Convert each question into 5 (prompt, option) pairs for multiple-choice format

In [ ]:
choice_map = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
choices = ["A", "B", "C", "D", "E"]

def preprocess_multiple_choice(examples):
    first_sentences = [[prompt] * 5 for prompt in examples["prompt"]]
    second_sentences = [
        [examples[choice][i] for choice in choices]
        for i in range(len(examples["prompt"]))
    ]

    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized = tokenizer(
        first_sentences,
        second_sentences,
        truncation=True,
        max_length=384,
        padding="max_length"
    )
    return {k: [v[i:i + 5] for i in range(0, len(v), 5)] for k, v in tokenized.items()}

### Build train/val datasets

In [ ]:
train_df["label"] = train_df["answer"].map(choice_map)
val_df["label"] = val_df["answer"].map(choice_map)

train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)

train_ds = train_ds.map(preprocess_multiple_choice, batched=True)
val_ds = val_ds.map(preprocess_multiple_choice, batched=True)

drop_cols = ["prompt", "A", "B", "C", "D", "E", "answer"]
train_ds = train_ds.remove_columns(drop_cols)
val_ds = val_ds.remove_columns(drop_cols)

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

### Load model

In [ ]:
model = AutoModelForMultipleChoice.from_pretrained(model_name,torch_dtype=torch.float32)
model = model.cuda()

### Adapter
Trainer passes logits + labels as one object, so we unpack them here before calling scorer.

In [ ]:
def compute_metrics_for_trainer(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    top3_preds = np.argsort(logits, axis=-1)[:, ::-1][:, :3]

    score_map3 = mean_average_precision_at_3(labels, top3_preds)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")

    return {"map3": score_map3, "accuracy": acc, "f1": f1}

### Training arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./deberta_base",
    learning_rate=1e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    num_train_epochs=10,
    logging_steps=10,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="map3",
    greater_is_better=True,
    fp16=True,
    bf16=False,
    report_to="wandb",  
)

### Train

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics_for_trainer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

# Predict on Test Set

In [ ]:
test_ds = Dataset.from_pandas(test_df)
test_ds = test_ds.map(preprocess_multiple_choice, batched=True)
test_ds = test_ds.remove_columns(["prompt", "A", "B", "C", "D", "E"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask"])

test_preds = trainer.predict(test_ds)
logits = test_preds.predictions

top3_indices = np.argsort(logits, axis=-1)[:, ::-1][:, :3]
test_predictions = [" ".join([choices[i] for i in row]) for row in top3_indices]

# Submission Cell

In [ ]:
submission = pd.DataFrame({"ID": test_df["id"], "Prediction": test_predictions})
submission.to_csv("submission.csv", index=False)
submission.head()